# Laboratorio 3.3 - Amazon SageMaker: Codificación de datos categóricos

**Nombre completo:** Richard Santiago Caicedo Toro

## Objetivo
Aplicar técnicas de codificación (encoding) a las variables categóricas del dataset para
dejarlo listo como insumo de un futuro modelo de Machine Learning.

## Explicación de la tarea
Se identifican las variables categóricas del dataset (ya limpio, resultado del Lab 3.2) y
se aplican distintas técnicas de codificación según su naturaleza: Label Encoding para
variables ordinales, One-Hot Encoding para variables nominales con pocas categorías, y se
comenta cuándo sería apropiado usar otras técnicas (p. ej. Target/Frequency Encoding) para
variables de alta cardinalidad.


## 1. Importar librerías

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## 2. Cargar el dataset limpio (resultado del Lab 3.2)

In [9]:
data_path = Path("data/dataset.csv")
if not data_path.exists():
    data_path = Path("../data/dataset.csv")

if not data_path.exists():
    raise FileNotFoundError("No se encontró data/dataset.csv ni ../data/dataset.csv")

df = pd.read_csv(data_path).drop_duplicates().copy()
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include="str").columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Dataset limpio cargado desde: {data_path}")
df.head()


Dataset limpio cargado desde: ..\data\dataset.csv


,student_id,age,city,study_hours,attendance,performance_level,final_score
0,1,20,Bogota,8.0,92.0,alto,4.6
1,2,21,Medellin,5.0,85.0,medio,3.8
2,3,19,Cali,3.0,72.0,bajo,2.9
3,4,22,Bogota,7.0,88.0,medio,4.1
4,5,20,Cartagena,2.0,65.0,bajo,2.5


## 3. Identificar variables categóricas

In [10]:
cat_cols = df.select_dtypes(include="str").columns.tolist()
print("Variables categóricas encontradas:", cat_cols)

for col in cat_cols:
    print(f"\n{col}: {df[col].nunique()} categorías únicas")
    print(df[col].value_counts().head())


Variables categóricas encontradas: ['city', 'performance_level']

city: 4 categorías únicas
city
Bogota       4
Medellin     4
Cali         4
Cartagena    2
Name: count, dtype: int64

performance_level: 3 categorías únicas
performance_level
medio    7
alto     4
bajo     3
Name: count, dtype: int64


## 4. Label Encoding (variables ordinales o binarias)

Útil cuando existe un orden natural entre las categorías (p. ej. 'bajo', 'medio', 'alto')
o cuando la variable es binaria.

In [11]:
ordinal_cols = ["performance_level"]
ordinal_order = {"bajo": 0, "medio": 1, "alto": 2}

for col in ordinal_cols:
    if col in df.columns:
        df[f"{col}_encoded"] = df[col].map(ordinal_order)

df[[col for col in ordinal_cols if col in df.columns] + [f"{col}_encoded" for col in ordinal_cols if col in df.columns]].head()


,performance_level,performance_level_encoded
0,alto,2
1,medio,1
2,bajo,0
3,medio,1
4,bajo,0


## 5. One-Hot Encoding (variables nominales sin orden)

Útil cuando las categorías no tienen un orden implícito y su número no es demasiado
grande, para evitar crear un orden artificial que el modelo pueda malinterpretar.

In [12]:
nominal_cols = [col for col in ["city"] if col in df.columns]
df_encoded = df.drop(columns=[col for col in ordinal_cols if col in df.columns]).copy()
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=True, dtype=int)
df_encoded.head()


,student_id,age,study_hours,attendance,final_score,performance_level_encoded,city_Cali,city_Cartagena,city_Medellin
0,1,20,8.0,92.0,4.6,2,0,0,0
1,2,21,5.0,85.0,3.8,1,0,0,1
2,3,19,3.0,72.0,2.9,0,1,0,0
3,4,22,7.0,88.0,4.1,1,0,0,0
4,5,20,2.0,65.0,2.5,0,0,1,0


## 6. Verificación del resultado

In [13]:
print("Dimensiones antes del encoding:", df.shape)
print("Dimensiones después del encoding:", df_encoded.shape)
df_encoded.dtypes


Dimensiones antes del encoding: (14, 8)
Dimensiones después del encoding: (14, 9)


student_id                     int64
age                            int64
study_hours                  float64
attendance                   float64
final_score                  float64
performance_level_encoded      int64
city_Cali                      int64
city_Cartagena                 int64
city_Medellin                  int64
dtype: object